> **Legacy runbook notice (2026-08-13)**
>
> 이 통합 노트북은 과거 Phase 0--3 실행 기록을 보존한다. 최신 corrected protocol, pair-local H0--H3, action-alignment, weak-label QA는 `notebooks/README.md`의 공식 Phase 0/1A/2A/2R/2D/3A/3B 노트북을 사용한다. 이 파일의 기존 셀과 결과 경로는 재현성 때문에 유지한다.

# Graph-CLaD canonical Colab runbook

실행 순서: Phase 0 → Phase 1 → Phase 2A/2D QA → Phase 3. 핵심 구현은 `scripts/phase*/`에서 불러오며 노트북 셀에 복사하지 않는다. Phase 2R은 진단 전용이고 Phase 4는 Phase 3 control gate 이후에만 시작한다.

## 0. Drive와 로컬 프로젝트 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path('/content/Graph-CLaD')
if not (PROJECT_ROOT / 'scripts').exists():
    raise FileNotFoundError('Clone or upload Graph-CLaD to /content/Graph-CLaD first')
sys.path.insert(0, str(PROJECT_ROOT))
%cd {PROJECT_ROOT}

In [ ]:
import torch
runtime = {
    'cuda_available': torch.cuda.is_available(),
    'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'torch': torch.__version__,
}
runtime

## Phase 0 — supplied CLaD baseline smoke

In [ ]:
!python -m scripts.phase0.smoke --config configs/phase0_synthetic.json --device {'cuda' if torch.cuda.is_available() else 'cpu'}

## Phase 1 / Phase 2A — state와 graph 계약 검사

In [ ]:
!python -m unittest tests/test_phase1_inspection.py tests/test_phase2_graph_extractor.py

## Phase 2D — 검증 완료된 official-demo release 확인

새 데이터 생성은 `python -m scripts.phase2d.build_demo_dataset --help`의 resume 가능한 변환기를 사용한다. 아래 셀은 기존 input-clean release에서 Phase 3 smoke 입력만 만든다.

In [ ]:
from scripts.phase3.dataset_io import build_smoke_dataset

PHASE2D_ROOT = Path('/content/drive/MyDrive/Graph-CLaD/artifacts/phase2d/data/phase2d_full_demo_v2_inputclean_stream1')
PHASE2D_FILES = [
    PHASE2D_ROOT / f'task{task_id}/phase2d_task{task_id}_graph_dataset.jsonl.gz'
    for task_id in (0, 1, 2)
]
missing = [str(path) for path in PHASE2D_FILES if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing persistent Phase 2D files: {missing}')
smoke = build_smoke_dataset(PHASE2D_FILES, per_split=100)
smoke_path = Path('/content/drive/MyDrive/Graph-CLaD/artifacts/phase3/phase3_smoke_dataset.json')
smoke_path.parent.mkdir(parents=True, exist_ok=True)
smoke_path.write_text(json.dumps(smoke, ensure_ascii=False), encoding='utf-8')
{'path': str(smoke_path), 'split_counts': smoke['split_counts']}

## Phase 2D — target-aligned holding dataset QA

기존 official-demo input-clean release에서 생성한 holding target artifact를 검사한다. Dataset을 다시 생성해야 할 때만 `python -m scripts.phase2d.build_holding_target_dataset --help`를 사용한다.

In [ ]:
HOLDING_TARGET_ROOT = Path('/content/drive/MyDrive/Graph-CLaD/artifacts/phase2d/data/phase2d_holding_target_v2_inputclean_stream1')
HOLDING_TARGET_MANIFEST = HOLDING_TARGET_ROOT / 'phase2d_holding_target_dataset_manifest.json'
if not HOLDING_TARGET_MANIFEST.exists():
    raise FileNotFoundError(HOLDING_TARGET_MANIFEST)
holding_manifest = json.loads(HOLDING_TARGET_MANIFEST.read_text(encoding='utf-8'))
if holding_manifest.get('status') != 'pass':
    raise RuntimeError(holding_manifest.get('warnings'))
audit_path = HOLDING_TARGET_ROOT / 'phase2d_holding_target_sampler_audit.json'
!python -m scripts.phase2d.audit_holding_target_dataset --dataset-root {HOLDING_TARGET_ROOT} --output {audit_path} --cap 600 --category-quota 120

## Phase 3 — GNN smoke

Smoke는 배선 확인용이다. Phase 4로 가기 전 full run에서 correct action, no-action, shuffled-action, shuffled-edge를 모두 비교해야 한다.

In [ ]:
smoke_config = {
    'probe_version': 'phase3-offline.v1-smoke',
    'models': ['p2_gnn_empty_edge'],
    'seeds': [0],
    'hidden_dim': 64,
    'batch_size': 64,
    'epochs': 3,
    'patience': 2,
    'learning_rate': 0.001,
    'current_loss_weight': 0.25,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}
config_path = smoke_path.with_name('phase3_smoke_config.json')
output_path = smoke_path.with_name('phase3_smoke_result.json')
config_path.write_text(json.dumps(smoke_config, indent=2), encoding='utf-8')
!python -m scripts.phase3.offline_probe --dataset {smoke_path} --config {config_path} --output {output_path}

## Phase 3 — holding challenge controlled run

`balanced_v3` config는 기존 Colab 결과의 정확한 sampler 재현용이다. 새 실험은 `balanced_v4_samplingfix`를 사용한다. 두 protocol 결과를 한 평균으로 합치지 않는다.

In [ ]:
RUN_FULL_HOLDING_EXPERIMENT = False
holding_config = PROJECT_ROOT / 'configs/phase3_holding_target_balanced_v4_samplingfix.json'
holding_output = Path('/content/drive/MyDrive/Graph-CLaD/artifacts/phase3_holding_target_balanced_v4_samplingfix')
if RUN_FULL_HOLDING_EXPERIMENT:
    !python -m scripts.phase3.run_controlled_taskfamily --dataset-root {HOLDING_TARGET_ROOT} --output-root {holding_output} --max-samples-per-family 600 --config {holding_config}
else:
    print('Full run skipped; set RUN_FULL_HOLDING_EXPERIMENT=True after reviewing the config.')

## Phase 3 — saved holding report analysis

In [ ]:
saved_report = Path('/content/drive/MyDrive/Graph-CLaD/artifacts/phase3_holding_target_balanced_v3/phase3_controlled_taskfamily_report.json')
saved_analysis = saved_report.with_name('phase3_holding_target_balanced_analysis_localized.json')
if saved_report.exists():
    !python -m scripts.phase3.analyze_holding_results --report {saved_report} --output {saved_analysis} --relation holding
else:
    print('Saved report is missing:', saved_report)